In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Summary

Construct theory of mind dataset(s) for RL.

In [51]:
from collections import Counter, defaultdict
import os
import json
from pathlib import Path
import string
import numpy as np
import pandas as pd
import random
from typing import Optional, Union, Any
from datasets import Dataset

from aeon.datasets import save_dataset
from aeon import config

## Theory of Mind bench

https://github.com/zhchen18/ToMBench/tree/main

In [11]:
tom_bench_path = config.PROJECT_ROOT.parent/"ToMBench/data"

In [62]:
def load_line(line: str):
    data = json.loads(line)
    res = {}
    for k, v in data.items():
        *_, key = k.split("\n")
        if key.lower() in "ABCD" or key[0] not in string.ascii_letters:
            continue
        res[key.replace('-', '_').lower()] = v
    return res

In [63]:
# Map name to list[dict].
datasets = {}
for path in tom_bench_path.iterdir():
    if path.suffix != ".jsonl":
        continue
    with open(path, "r") as f:
        datasets[path.stem.lower().replace(' ', '_').replace('-', '_')] = [
            load_line(line) for line in f
        ]

In [64]:
sorted(
    [(k, len(v)) for k, v in datasets.items()],
    key=lambda x: x[-1],
    reverse=True
)

[('false_belief_task', 600),
 ('faux_pas_recognition_test', 560),
 ('strange_story_task', 407),
 ('unexpected_outcome_test', 300),
 ('ambiguous_story_task', 200),
 ('scalar_implicature_test', 200),
 ('hinting_task_test', 103),
 ('persuasion_story_task', 100),
 ('hidden_emotions', 80),
 ('moral_emotions', 40),
 ('percepts_knowledge_links', 40),
 ('discrepant_emotions', 40),
 ('discrepant_intentions', 40),
 ('knowledge_pretend_play_links', 30),
 ('discrepant_desires', 20),
 ('multiple_desires', 20),
 ('prediction_of_actions', 20),
 ('knowledge_attention_links', 20),
 ('emotion_regulation', 20),
 ('completion_of_failed_actions', 20)]

In [65]:
keys = defaultdict(int)
for ds in datasets.values():
    for key in ds[0]:
        keys[key] += 1

In [66]:
sorted(keys.items(), key=lambda x: x[1])

[('ability', 20),
 ('index', 20),
 ('story', 20),
 ('question', 20),
 ('option_a', 20),
 ('option_b', 20),
 ('option_c', 20),
 ('option_d', 20),
 ('answer', 20)]

In [82]:
df = pd.concat(
    [
        pd.DataFrame(ds).assign(
            dataset=name,
            dataset_parent=lambda x: x.ability.str.partition(':', expand=False)
                                       .str[0].str.lower().str.replace(' ', '_')
        )
        for name, ds in datasets.items()
    ],
    axis=0
)

In [116]:
row = df.sample().iloc[0]
print(row.dataset, end='\n\n')
print('Story:', row.story, end='\n\n')
print('Q:', row.question, end='\n\n')
print(row[[c for c in row.index if c.startswith('option')]].to_dict(), end='\n\n')
print(row.answer)

discrepant_intentions

Story: James shows his painting at the school art exhibition. One day, he forgets to take the painting home. Nick, while cleaning the exhibition room, mistakes the painting for an abandoned one and puts it in the discard box. At the same time, Ella knows this is James's work. Ella's own painting also shows at the art exhibition, and she does not want James's painting to be more popular than hers, so she decides not to tell Nick. When James finds the painting is missing, he feels very upset and starts asking classmates. Later, Nick realizes he may have mishandled James's painting.

Q: What is the possible intention behind Nick's action?

{'option_a': 'Nick possibly does not care about the painting, so he does not notice its value.', 'option_b': "Nick possibly acts unconsciously or unknowingly, as he mistakenly thinks James's work is abandoned.", 'option_c': 'Nick possibly thinks cleaning the exhibition room is more important than preserving the work.', 'option_d':

In [117]:
# Multiple choice task: generate letter only
# RL compatible; x1 rows
# TODO: confirm nanochate supports system msg
# TODO: maybe we could logit bias to constraint answres to A-D
mc_template = "STORY: {story}\nQUESTION: {question}\nOPTIONS: A. {option_a}\nB. {option_b}\nC. {option_c}\nD. {option_d}"
[
    {"role": "system", "content": "Answer with a single uppercase letter corresponding to the option you think is correct."},
    {"role": "user", "content": mc_template.format(**row.to_dict())},
    {"role": "assistant", "content": row.answer}
]

[{'role': 'system',
  'content': 'Answer with a single uppercase letter corresponding to the option you think is correct.'},
 {'role': 'user',
  'content': "STORY: James shows his painting at the school art exhibition. One day, he forgets to take the painting home. Nick, while cleaning the exhibition room, mistakes the painting for an abandoned one and puts it in the discard box. At the same time, Ella knows this is James's work. Ella's own painting also shows at the art exhibition, and she does not want James's painting to be more popular than hers, so she decides not to tell Nick. When James finds the painting is missing, he feels very upset and starts asking classmates. Later, Nick realizes he may have mishandled James's painting.\nQUESTION: What is the possible intention behind Nick's action?\nOPTIONS: A. Nick possibly does not care about the painting, so he does not notice its value.\nB. Nick possibly acts unconsciously or unknowingly, as he mistakenly thinks James's work is aband

In [119]:
# FREE RESPONSE TASK: generate correct free text answer
# less rl-compatible; 1x rows
# TODO: still considering how this framing would work. Could use in rl and require
# exact match, minus capitalization; could use during mid or chat_sft training; could
# create more of an RLHF dataset with higher scores for correct answers (or the reverse 😈)
template = "STORY: {story}\nQUESTION: {question}"
[
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "assistant", "content": row[f'option_{row.answer.lower()}']}
]

[{'role': 'user',
  'content': "STORY: James shows his painting at the school art exhibition. One day, he forgets to take the painting home. Nick, while cleaning the exhibition room, mistakes the painting for an abandoned one and puts it in the discard box. At the same time, Ella knows this is James's work. Ella's own painting also shows at the art exhibition, and she does not want James's painting to be more popular than hers, so she decides not to tell Nick. When James finds the painting is missing, he feels very upset and starts asking classmates. Later, Nick realizes he may have mishandled James's painting.\nQUESTION: What is the possible intention behind Nick's action?"},
 {'role': 'assistant',
  'content': "Nick possibly acts unconsciously or unknowingly, as he mistakenly thinks James's work is abandoned."}]

In [125]:
# BINARY TASK: mark user answer as correct/incorrect
# rl compatible; potentially 4x rows
# TODO: confirm nanochat handles system messages
option = random.choice([c for c in row.index if c.startswith('option')])
answer = row[option]
label = str(int(option.split('_')[-1] == row.answer.lower()))
[
    {"role": "system", "content": "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
    {"role": "user", "content": template.format(**row.to_dict())},
    {"role": "user", "content": f"ANSWER: {answer}"},
    {"role": "assistant", "content": label}
]

[{'role': 'system',
  'content': "Generate a single integer (0 or 1) grading the user's answer as correct or incorrect."},
 {'role': 'user',
  'content': "STORY: James shows his painting at the school art exhibition. One day, he forgets to take the painting home. Nick, while cleaning the exhibition room, mistakes the painting for an abandoned one and puts it in the discard box. At the same time, Ella knows this is James's work. Ella's own painting also shows at the art exhibition, and she does not want James's painting to be more popular than hers, so she decides not to tell Nick. When James finds the painting is missing, he feels very upset and starts asking classmates. Later, Nick realizes he may have mishandled James's painting.\nQUESTION: What is the possible intention behind Nick's action?"},
 {'role': 'user',
  'content': 'ANSWER: Nick possibly does not care about the painting, so he does not notice its value.'},
 {'role': 'assistant', 'content': '0'}]

## Higher order theory of mind dataset

https://github.com/ying-hui-he/Hi-ToM_dataset/tree/main

In [131]:
hi_tom_path = config.DATA_DIR/"raw/hi-tom/hi-tom.json"

In [133]:
with open(hi_tom_path, "r") as f:
    hi_tom = json.load(f)

In [141]:
hi_tom['data'][0].keys()

dict_keys(['prompting_type', 'deception', 'story_length', 'question_order', 'sample_id', 'story', 'question', 'choices', 'answer', 'prompt'])

In [143]:
df_hi = pd.DataFrame(hi_tom['data'])

In [145]:
df_hi.prompting_type.value_counts()

prompting_type
CoTP    600
VP      600
Name: count, dtype: int64

In [146]:
df_hi.deception.value_counts()

deception
False    600
True     600
Name: count, dtype: int64

In [148]:
df_hi.question_order.value_counts()

question_order
0    240
1    240
2    240
3    240
4    240
Name: count, dtype: int64

In [153]:
hi_row = df_hi.sample().iloc[0]

In [157]:
# TODO: could also put prompt in system message, or split it into instructions in system message
# and story + answer options in user message?
[
    {"role": "user", "content": hi_row.prompt.replace(
        "answer the multiple-choice question", "answer the multiple-choice question with a single snake_case word not including the preceding choice letter"
    )},
    {"role": "assistant", "content": hi_row.answer}
]

[{'role': 'user',
  'content': "Read the following story and answer the multiple-choice question with a single snake_case word not including the preceding choice letter. Think step-by-step. Provide the answer first, and then explain it.\nStory:\n1 Liam, Noah, Avery, Mila and Benjamin entered the workshop.\n2 The peas is in the green_drawer.\n3 Liam moved the peas to the blue_container.\n4 Liam dislikes the peas.\n5 Liam exited the workshop.\n6 Noah moved the peas to the green_crate.\n7 Noah exited the workshop.\n8 Avery moved the peas to the blue_bucket.\n9 Avery exited the workshop.\n10 Mila moved the peas to the blue_crate.\n11 Liam saw a dog.\n12 Mila exited the workshop.\n13 Benjamin made no movements and stayed in the workshop for 1 minute.\n14 Benjamin exited the workshop.\n15 Liam, Noah, Avery, Mila and Benjamin entered the waiting_room.\n16 Noah, Liam and Mila entered the den.\n17 The melon is in the red_bucket.\n18 Noah made no movements and stayed in the den for 1 minute.\n19